In [1]:
import json
import pickle

import numpy as np

from typing import Any, Dict, List, Literal, Optional

from llm_foundation import logger
from llm_foundation.agent_types import Persona, Role

from crewai import Agent, Task, Crew
from crewai.crews import CrewOutput
from hackathon.index import generate_embeddings, create_index, search_index, build_similar_entities
from hackathon.input_output_types import NamedEntities
from hackathon.retrieval_neo4j import chunk_ranker, retrieve_context, retrieve_similar_entities, pagerank
from hackathon.graph_neo4j import add_entities, add_relates_to_relationships, build_vector_index, add_similar_entities, clean_db, Neo4jClientFactory
from hackathon.tools import filter_named_entities, create_document_deduped_entities_dict, create_matrix_entity_ref_count
from hackathon.utils import build_document_structure, save_document_structure
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
 
from langchain.output_parsers.json import SimpleJsonOutputParser
from rich import print
from rich.pretty import pprint


from dotenv import load_dotenv

load_dotenv()

2024-12-02 15:10:52 R2GWRJJGF9 root[75898] INFO Logger root configured


Trying to configure logger root in module llm_foundation
root # of associated handlers - 0
Logging is not configured yet. Configuring it now.
Basic logging config


2024-12-02 15:10:52 R2GWRJJGF9 flaml[75898] WARNING flaml.automl is not available. Please install flaml[automl] to enable AutoML functionalities.
2024-12-02 15:10:54 R2GWRJJGF9 httpx[75898] INFO HTTP Request: GET https://raw.githubusercontent.com/BerriAI/litellm/main/model_prices_and_context_window.json "HTTP/1.1 200 OK"
2024-12-02 15:10:54 R2GWRJJGF9 faiss.loader[75898] INFO Loading faiss.
2024-12-02 15:10:54 R2GWRJJGF9 faiss.loader[75898] INFO Successfully loaded faiss.


True

In [2]:
context_retrieval = Persona.from_yaml_file("../Personas/ContextRetrievalCrewAI.yaml")
context_retrieval

2024-12-02 15:10:56 R2GWRJJGF9 root[75898] INFO YAML data:
{'name': 'Context Retrieval', 'roles': {'document_structure_loader': {'name': 'Document Structure Loader', 'description': 'You are capable of finding and loading the document structure of a document.\n', 'agent_system_message': 'You are capable of finding and loading the document structure of a document.\n', 'tasks': {'load_document_structure': {'name': 'Load Document Structure', 'description': "From a file passed from the user, you're capable of loading the document chunks.\n\nDocument to load:\n```{document_name}```\n", 'expected_output': 'The structure of a document.\nThis includes:\n  - A list of objects describing the chunks of the document itself.\n'}}}, 'entity_retriever': {'name': 'Entity Retriever', 'description': 'You are capable of finding similar entities from a list of source entities.\n', 'agent_system_message': 'You are capable of finding similar entities from a list of source entities.\n', 'tasks': {'retrieve_si

Persona(name='Context Retrieval', roles={'document_structure_loader': Role(name='Document Structure Loader', description='You are capable of finding and loading the document structure of a document.\n', agent_system_message='You are capable of finding and loading the document structure of a document.\n', examples=[], tasks={'load_document_structure': Task(name='Load Document Structure', description="From a file passed from the user, you're capable of loading the document chunks.\n\nDocument to load:\n```{document_name}```\n", expected_output='The structure of a document.\nThis includes:\n  - A list of objects describing the chunks of the document itself.\n')}, human_input_mode='TERMINATE', autogen_code_execution_config={}), 'entity_retriever': Role(name='Entity Retriever', description='You are capable of finding similar entities from a list of source entities.\n', agent_system_message='You are capable of finding similar entities from a list of source entities.\n', examples=[], tasks={'

# Document loader

In [ ]:
pickle.loads(open(f"../2405.14831v1_document_structure_with_ne.pkl", "rb").read())

In [3]:
document_structure_loader_role: Role = context_retrieval.get_role("document_structure_loader")
pprint(document_structure_loader_role)
document_structure_loader: Agent = document_structure_loader_role.to_crewai_agent(verbose=True, allow_delegation=False) # type: ignore

from hackathon.tools import doc_structure_loader_tool, DocumentStructure
tools = [doc_structure_loader_tool]

load_document_structure: Task = document_structure_loader_role.get_crew_ai_task("load_document_structure", document_structure_loader, tools=tools, output_pydantic=DocumentStructure) # type: ignore

query_inputs = {
    "document_name": "../2405.14831v1.pkl",
}

crew = Crew(
    agents=[document_structure_loader],
    tasks=[load_document_structure],
    verbose=True,
)

logger.info("^^^^^^^^^^^^^^^^^^^^^^^^^^^^ Calling Agents ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^")
result: CrewOutput = crew.kickoff(inputs=query_inputs)
result

Role(
│   name='Document Structure Loader',
│   description='You are capable of finding and loading the document structure of a document.\n',
│   agent_system_message='You are capable of finding and loading the document structure of a document.\n',
│   examples=[],
│   tasks={
│   │   'load_document_structure': Task(
│   │   │   name='Load Document Structure',
│   │   │   description="From a file passed from the user, you're capable of loading the document chunks.\n\nDocument to load:\n```{document_name}```\n",
│   │   │   expected_output='The structure of a document.\nThis includes:\n  - A list of objects describing the chunks of the document itself.\n'
│   │   )
│   },
│   human_input_mode='TERMINATE',
│   autogen_code_execution_config={}
)

2024-12-02 15:11:00 R2GWRJJGF9 root[75898] INFO Creating CrewAI Task Load Document Structure
2024-12-02 15:11:00 R2GWRJJGF9 root[75898] INFO ^^^^^^^^^^^^^^^^^^^^^^^^^^^^ Calling Agents ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
15:11:00 - LiteLLM:INFO: utils.py:2749 - 
LiteLLM completion() model= gpt-4o-mini; provider = openai


Checking env var OPENAI_API_KEY: sk-svcacct-ac9FyfFdCc6qJCxmRHGbwULjD6-ZJyqJ5LMFk6N0UWV3cZyg86k2N87_itz1ST3BlbkFJVB8Ki3M3gW2zFetrawhpIEfoclaX_wtKZ7axzIsCa08ZTsiopVk6--2RuaTKAA
Mapped key name: api_key
# Agent: Document Structure Loader
## Task: From a file passed from the user, you're capable of loading the document chunks.

Document to load:
```../2405.14831v1.pkl```



15:11:01 - LiteLLM:INFO: utils.py:944 - Wrapper: Completed Call, calling success_handler
15:11:01 - LiteLLM:INFO: utils.py:2749 - 
LiteLLM completion() model= gpt-4o-mini; provider = openai




# Agent: Document Structure Loader
## Thought: I need to load the document structure of the provided file to retrieve the required chunks of the document.
## Using tool: doc_structure_loader_tool
## Tool Input: 
"{\"document_name\": \"../2405.14831v1.pkl\"}"
## Tool Output: 
[{'id': 0, 'text': 'HippoRAG: Neurobiologically Inspired\nLong-Term Memory for Large Language Models\nBernal Jiménez Gutiérrez\nThe Ohio State University\njimenezgutierrez.1@osu.eduYiheng Shu\nThe Ohio State University\nshu.251@osu.edu\nYu Gu\nThe Ohio State University\ngu.826@osu.eduMichihiro Yasunaga\nStanford University\nmyasu@cs.stanford.eduYu Su\nThe Ohio State University\nsu.809@osu.edu\nAbstract\nIn order to thrive in hostile and ever-changing natural environments, mammalian\nbrains evolved to store large amounts of knowledge about the world and continually\nintegrate new information while avoiding catastrophic forgetting. Despite the\nimpressive accomplishments, large language models (LLMs), even with ret

15:11:52 - LiteLLM:INFO: utils.py:944 - Wrapper: Completed Call, calling success_handler




# Agent: Document Structure Loader
## Final Answer: 
```json
[
    {
        "id": 0,
        "text": "HippoRAG: Neurobiologically Inspired\nLong-Term Memory for Large Language Models...",
        "named_entities": ["stanford university", "yiheng shu", "yu su", "bernal jiménez gutiérrez", "hipporag", "yu gu", "michihiro yasunaga", "neurobiologically inspired", "long-term memory", "the ohio state university", "large language models"],
        "triples": [["HippoRAG", "is", "Neurobiologically Inspired"], ["HippoRAG", "is a framework for", "Long-Term Memory"], ["HippoRAG", "is designed for", "Large Language Models"], ["Bernal Jiménez Gutiérrez", "affiliated with", "The Ohio State University"], ["Yiheng Shu", "affiliated with", "The Ohio State University"], ["Yu Gu", "affiliated with", "The Ohio State University"], ["Michihiro Yasunaga", "affiliated with", "Stanford University"], ["Yu Su", "affiliated with", "The Ohio State University"], ["The Ohio State University", "has researchers", "

15:11:53 - LiteLLM:INFO: utils.py:2749 - 
LiteLLM completion() model= gpt-4o-mini; provider = openai
2024-12-02 15:11:53 R2GWRJJGF9 LiteLLM[75898] INFO 
LiteLLM completion() model= gpt-4o-mini; provider = openai
2024-12-02 15:11:59 R2GWRJJGF9 httpx[75898] INFO HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"
15:11:59 - LiteLLM:INFO: utils.py:944 - Wrapper: Completed Call, calling success_handler
2024-12-02 15:11:59 R2GWRJJGF9 LiteLLM[75898] INFO Wrapper: Completed Call, calling success_handler


CrewOutput(raw='```json\n[\n    {\n        "id": 0,\n        "text": "HippoRAG: Neurobiologically Inspired\\nLong-Term Memory for Large Language Models...",\n        "named_entities": ["stanford university", "yiheng shu", "yu su", "bernal jiménez gutiérrez", "hipporag", "yu gu", "michihiro yasunaga", "neurobiologically inspired", "long-term memory", "the ohio state university", "large language models"],\n        "triples": [["HippoRAG", "is", "Neurobiologically Inspired"], ["HippoRAG", "is a framework for", "Long-Term Memory"], ["HippoRAG", "is designed for", "Large Language Models"], ["Bernal Jiménez Gutiérrez", "affiliated with", "The Ohio State University"], ["Yiheng Shu", "affiliated with", "The Ohio State University"], ["Yu Gu", "affiliated with", "The Ohio State University"], ["Michihiro Yasunaga", "affiliated with", "Stanford University"], ["Yu Su", "affiliated with", "The Ohio State University"], ["The Ohio State University", "has researchers", "Bernal Jiménez Gutiérrez"], ["Th

In [4]:
doc: DocumentStructure = result.pydantic

In [9]:
len(doc.structure)

1

# Entity Retriever

In [ ]:
entity_retriever_role: Role = context_retrieval.get_role("entity_retriever")

entity_retriever: Agent = entity_retriever_role.to_crewai_agent(verbose=True, allow_delegation=False) # type: ignore

from hackathon.tools import similar_entities_retrieval_tool

tools = [similar_entities_retrieval_tool]

class Entities(BaseModel):
    entities: List[str] = Field(..., description="List of entities")
    
retrieve_similar_entities: Task = entity_retriever_role.get_crew_ai_task("retrieve_similar_entities", entity_retriever, tools=tools, output_pydantic=Entities) # type: ignore

query_inputs = {
    "entities": ["bernal"],
}

crew = Crew(
    agents=[entity_retriever],
    tasks=[retrieve_similar_entities],
    verbose=True,
)

logger.info("^^^^^^^^^^^^^^^^^^^^^^^^^^^^ Calling Agents ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^")
result: CrewOutput = crew.kickoff(inputs=query_inputs)
result


In [ ]:
result.pydantic.structure